# Сравнение методов выявления фейковых новостей (русскоязычный корпус)
**Корпус синтетический** (шаблонная генерация, 3200 текстов, 4 темы, метки корректны, классы различаются вероятностями стилевых маркеров). В статье его нужно прямо называть синтетическим; для реальных данных замените `ru_fake_news_synthetic.csv` (колонки `title,text,label,domain`). label: 1 = достоверная, 0 = фейк.

**Протокол:** стратифицированное 80/20; 5-fold CV и тесты значимости — на train; итог — один раз на отложенном test.

- **A** (выполнена): BoW, TF-IDF, NB, LogReg, SVM, RF, градиентный бустинг, Word2Vec/FastText, MLP, текст+метаданные, XAI (окклюзия слов, перестановочная важность).
- **B** (`RUN_DL=True`, нужны torch/transformers, лучше GPU): LSTM, GRU, CNN, BERT, BERT-CNN, BERT-LSTM.
- **C** (`RUN_LLM=True`): zero-shot LLM.

In [ ]:
import os, re, time, random, warnings, itertools
import numpy as np, pandas as pd
from scipy import stats
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix
from gensim.models import Word2Vec, FastText
warnings.filterwarnings('ignore'); SEED=42; N_SPLITS=5; PATH='ru_fake_news_synthetic.csv'; RUN_DL=True
os.makedirs('output',exist_ok=True)

In [7]:
def generate(path,n=3200,seed=7):
    rng=random.Random(seed)
    topics={'политика':(['правительство','парламент','министр','закон','выборы','губернатор','комиссия','законопроект'],['принял','обсудил','внёс','одобрил','отклонил','рассмотрел']),
    'здоровье':(['врачи','вакцина','лекарство','больница','пациенты','диета','вирус','клиника'],['рекомендовали','изучили','выявили','назначили','испытали','зарегистрировали']),
    'экономика':(['банк','курс рубля','инфляция','бюджет','компания','рынок','налоги','экспорт'],['вырос','снизился','повысил','объявил','сообщил о','оценил']),
    'спорт':(['сборная','тренер','чемпионат','клуб','матч','игрок','турнир','стадион'],['выиграла','проиграл','подписал','объявил состав','вышла в финал','отстранили'])}
    fakeM=['ШОК','сенсация','от нас скрывают','срочно перешлите всем','вы не поверите','врачи в ужасе','правда, о которой молчат','разоблачение','тайный заговор','слив информации','это запретили показывать']
    realM=['сообщила пресс-служба','по данным ведомства','как отмечают эксперты','говорится в официальном заявлении','передаёт агентство','заявил представитель','согласно статистике','в рамках официальной проверки']
    neutral=['в понедельник','в этом году','по словам источников','в регионе','на прошлой неделе','в столице','за последний месяц','около 40 процентов','в ближайшее время','несколько человек']
    def sent(t):
        nn,vv=topics[t]; return f"{rng.choice(nn).capitalize()} {rng.choice(vv)} {rng.choice(nn)} {rng.choice(neutral)}."
    rows=[]
    for _ in range(n):
        t=rng.choice(list(topics)); y=int(rng.random()<.5); pf,pr=(.05,.40) if y else (.30,.15); ss=[]
        for _ in range(rng.randint(2,12)):
            s=sent(t)
            if rng.random()<pf: s=rng.choice(fakeM)+('!' if rng.random()<.6 else '')+' '+s
            if rng.random()<pr: s+=' '+rng.choice(realM).capitalize()+'.'
            ss.append(s)
        rows.append((sent(t)[:-1],' '.join(ss),y,t))
    pd.DataFrame(rows,columns=['title','text','label','domain']).to_csv(path,index=False)
if not os.path.exists(PATH): generate(PATH)
df_all=pd.read_csv(PATH); df_all['content']=df_all.title.fillna('')+' . '+df_all.text.fillna('')
n0=len(df_all); df_all=df_all[df_all.content.str.len()>20].drop_duplicates('content').reset_index(drop=True); print('удалено пустых/дублей:',n0-len(df_all))
df_all['n_words']=df_all.content.str.split().str.len(); df_all['n_excl']=df_all.content.str.count('!')
df_all['caps_ratio']=df_all.content.apply(lambda s: sum(w.isupper() and len(w)>1 for w in s.split())/max(1,len(s.split()))); META=['n_words','n_excl','caps_ratio']
tr_df,te_df=train_test_split(df_all,test_size=0.2,stratify=df_all.label,random_state=SEED); tr_df=tr_df.reset_index(drop=True); te_df=te_df.reset_index(drop=True)
y,yte=tr_df.label.values,te_df.label.values; print('train',len(tr_df),'test',len(te_df))

удалено пустых/дублей: 0
train 2560 test 640


## 1. Модели
Все принимают DataFrame. BoW и TF-IDF используются с одними классификаторами.

In [8]:
tok=lambda s: re.findall(r'\w+',s.lower())
class EmbAvg(BaseEstimator,TransformerMixin):
    def __init__(self,kind='w2v',dim=100): self.kind=kind; self.dim=dim
    def fit(self,X,y=None):
        S=[tok(s) for s in X['content']]; cls=Word2Vec if self.kind=='w2v' else FastText
        self.m_=cls(S,vector_size=self.dim,window=5,min_count=2,epochs=15 if self.kind=='w2v' else 6,seed=SEED,workers=1); return self
    def transform(self,X):
        out=np.zeros((len(X),self.dim))
        for i,s in enumerate(X['content']):
            v=[self.m_.wv[w] for w in tok(s) if w in self.m_.wv]; out[i]=np.mean(v,0) if v else 0
        return out
class SafeSVD(TruncatedSVD):
    def fit_transform(self,X,y=None):
        self.n_components=max(2,min(self.n_components,X.shape[1]-1,X.shape[0]-1)); return super().fit_transform(X,y)
def text(vec): return ColumnTransformer([('t',vec,'content')])
def tfidf(): return TfidfVectorizer(ngram_range=(1,2),min_df=3,max_features=100000,sublinear_tf=True)
def bow(): return CountVectorizer(ngram_range=(1,2),min_df=3,max_features=100000)
MODELS={
 'BoW+NB':lambda: make_pipeline(text(bow()),MultinomialNB(alpha=0.1)),
 'BoW+LogReg':lambda: make_pipeline(text(bow()),LogisticRegression(C=1,max_iter=2000,solver='liblinear')),
 'TFIDF+NB':lambda: make_pipeline(text(tfidf()),MultinomialNB(alpha=0.1)),
 'TFIDF+LogReg':lambda: make_pipeline(text(tfidf()),LogisticRegression(C=10,max_iter=1000,solver='liblinear')),
 'TFIDF+LinearSVM':lambda: make_pipeline(text(tfidf()),LinearSVC(C=1.0)),
 'TFIDF+RandomForest':lambda: make_pipeline(text(TfidfVectorizer(min_df=5,max_features=20000,sublinear_tf=True)),RandomForestClassifier(n_estimators=150,n_jobs=-1,random_state=SEED)),
 'TFIDF+GradBoost':lambda: make_pipeline(text(make_pipeline(TfidfVectorizer(min_df=5,max_features=20000,sublinear_tf=True),SafeSVD(100,random_state=SEED))),HistGradientBoostingClassifier(random_state=SEED)),
 'Word2Vec+LogReg':lambda: make_pipeline(EmbAvg('w2v'),StandardScaler(),LogisticRegression(max_iter=1000)),
 'FastText+LogReg':lambda: make_pipeline(EmbAvg('ft'),StandardScaler(),LogisticRegression(max_iter=1000)),
 'FastText+MLP':lambda: make_pipeline(EmbAvg('ft'),StandardScaler(),MLPClassifier((128,64),early_stopping=True,max_iter=300,random_state=SEED)),
 'TFIDF+MLP':lambda: make_pipeline(text(make_pipeline(TfidfVectorizer(min_df=3,max_features=20000,sublinear_tf=True),SafeSVD(150,random_state=SEED))),MLPClassifier((128,),early_stopping=True,max_iter=300,random_state=SEED)),
 'Text+Meta LogReg':lambda: make_pipeline(ColumnTransformer([('t',tfidf(),'content'),('m',StandardScaler(),META)]),LogisticRegression(C=10,max_iter=1000,solver='liblinear')),
}
SLICE=[k for k in MODELS if not k.startswith(('FastText','Word2Vec','TFIDF+RandomForest'))]  # для learning curve и LODO (быстрые модели)
def score(m,Xt): return m.predict_proba(Xt)[:,1] if hasattr(m,'predict_proba') else m.decision_function(Xt)
def metrics(yt,p,s):
    pr,rc,f1,_=precision_recall_fscore_support(yt,p,average='macro'); return dict(acc=accuracy_score(yt,p),prec=pr,rec=rc,f1=f1,auc=roc_auc_score(yt,s))

## 2. Кросс-валидация на train

In [9]:
folds=list(StratifiedKFold(N_SPLITS,shuffle=True,random_state=SEED).split(tr_df,y)); res=[]; oof={k:np.zeros(len(y),int) for k in MODELS}
for name,mk in MODELS.items():
    for f,(a,b) in enumerate(folds):
        t=time.time(); m=mk().fit(tr_df.iloc[a],y[a]); tt=time.time()-t; p=m.predict(tr_df.iloc[b])
        res.append(dict(model=name,fold=f,train_s=tt,**metrics(y[b],p,score(m,tr_df.iloc[b])))); oof[name][b]=p
R=pd.DataFrame(res); R.to_csv('output/cv_folds.csv',index=False)
summ=R.drop(columns='fold').groupby('model').agg(['mean','std']).round(4).sort_values(('acc','mean'),ascending=False); summ.to_csv('output/cv_summary.csv'); summ[[('acc','mean'),('acc','std'),('f1','mean'),('auc','mean'),('train_s','mean')]]

acc              f1     auc train_s
                      mean     std    mean    mean    mean
model                                                     
TFIDF+MLP           0.8531  0.0175  0.8531  0.9324  1.5547
TFIDF+GradBoost     0.8473  0.0285  0.8472  0.9300  2.4277
TFIDF+LogReg        0.8457  0.0181  0.8456  0.9288  0.3363
Text+Meta LogReg    0.8453  0.0173  0.8453  0.9270  0.3575
TFIDF+LinearSVM     0.8422  0.0215  0.8421  0.9243  0.3118
TFIDF+NB            0.8402  0.0272  0.8402  0.9224  0.2896
TFIDF+RandomForest  0.8402  0.0130  0.8400  0.9224  0.7548
BoW+NB              0.8387  0.0292  0.8386  0.9207  0.2810
Word2Vec+LogReg     0.8344  0.0170  0.8343  0.9196  2.4037
BoW+LogReg          0.8324  0.0136  0.8323  0.9186  0.3742
FastText+LogReg     0.8238  0.0214  0.8237  0.9051  6.0421
FastText+MLP        0.7883  0.0134  0.7880  0.8729  7.4206

## 3. Статистическая оценка
Bootstrap-ДИ 95%, критерий Макнемара с поправкой Холма, тест Фридмана.

In [10]:
def boot_ci(pred,B=1000):
    rng=np.random.default_rng(SEED); n=len(y); a=[accuracy_score(y[i],pred[i]) for i in (rng.integers(0,n,n) for _ in range(B))]; return np.percentile(a,[2.5,97.5])
ci=pd.DataFrame({k:[accuracy_score(y,oof[k]),*boot_ci(oof[k])] for k in MODELS},index=['acc','ci_low','ci_high']).T.round(4).sort_values('acc',ascending=False); ci.to_csv('output/cv_accuracy_ci.csv'); display(ci)
def mcn(a,b):
    ca=oof[a]==y; cb=oof[b]==y; n01=int((ca&~cb).sum()); n10=int((~ca&cb).sum()); return 1.0 if n01+n10==0 else stats.binomtest(n01,n01+n10,0.5).pvalue
pairs=list(itertools.combinations(MODELS,2)); pv=np.array([mcn(a,b) for a,b in pairs]); order=np.argsort(pv); holm=np.empty_like(pv); run=0
for r,i in enumerate(order): run=max(run,(len(pv)-r)*pv[i]); holm[i]=min(1,run)
P=pd.DataFrame({'pair':[' vs '.join(p) for p in pairs],'p_raw':pv,'p_holm':holm}); P['significant']=P.p_holm<0.05; P=P.sort_values('p_holm').round(5); P.to_csv('output/mcnemar_holm.csv',index=False)
print('значимых пар:',int(P.significant.sum()),'из',len(P)); display(P[P.significant].head(15))
piv=R.pivot(index='fold',columns='model',values='f1'); print('Friedman:',stats.friedmanchisquare(*[piv[c] for c in piv]))

,acc,ci_low,ci_high
TFIDF+MLP,0.8531,0.8402,0.8672
TFIDF+GradBoost,0.8473,0.8324,0.8621
TFIDF+LogReg,0.8457,0.8320,0.8598
Text+Meta LogReg,0.8453,0.8316,0.8590
TFIDF+LinearSVM,0.8422,0.8281,0.8570
TFIDF+RandomForest,0.8402,0.8254,0.8551
TFIDF+NB,0.8402,0.8266,0.8543
BoW+NB,0.8387,0.8246,0.8531
Word2Vec+LogReg,0.8344,0.8203,0.8492
BoW+LogReg,0.8324,0.8180,0.8469


значимых пар: 13 из 66


,pair,p_raw,p_holm,significant
63,FastText+MLP vs TFIDF+MLP,0.00000,0.00000,True
53,TFIDF+GradBoost vs FastText+MLP,0.00000,0.00000,True
35,TFIDF+LogReg vs FastText+MLP,0.00000,0.00000,True
64,FastText+MLP vs Text+Meta LogReg,0.00000,0.00000,True
48,TFIDF+RandomForest vs FastText+MLP,0.00000,0.00000,True
42,TFIDF+LinearSVM vs FastText+MLP,0.00000,0.00000,True
57,Word2Vec+LogReg vs FastText+MLP,0.00000,0.00000,True
27,TFIDF+NB vs FastText+MLP,0.00000,0.00000,True
8,BoW+NB vs FastText+MLP,0.00000,0.00000,True
60,FastText+LogReg vs FastText+MLP,0.00000,0.00001,True


Friedman: FriedmanchisquareResult(statistic=np.float64(23.98461538461541), pvalue=np.float64(0.01279822013864534))


## 4. Итоговая оценка на отложенном test (один раз)

In [11]:
rows=[]; test_pred={}; fitted={}
for k,mk in MODELS.items():
    m=mk().fit(tr_df,y); fitted[k]=m; test_pred[k]=m.predict(te_df); rows.append(dict(model=k,**metrics(yte,test_pred[k],score(m,te_df))))
T=pd.DataFrame(rows).set_index('model').round(4).sort_values('acc',ascending=False); T.to_csv('output/test_metrics.csv'); display(T)
cm=[]
for k in MODELS:
    tn,fp,fn,tp=confusion_matrix(yte,test_pred[k]).ravel(); cm.append((k,tn,fp,fn,tp,round(fp/(fp+tn),3),round(fn/(fn+tp),3)))
CM=pd.DataFrame(cm,columns=['model','TN','FP','FN','TP','FPR','FNR']).set_index('model'); CM.to_csv('output/test_confusion.csv'); display(CM)

,acc,prec,rec,f1,auc
model,,,,,
TFIDF+MLP,0.8547,0.8560,0.8551,0.8546,0.9231
TFIDF+LogReg,0.8312,0.8312,0.8312,0.8312,0.9150
TFIDF+GradBoost,0.8281,0.8281,0.8282,0.8281,0.9185
TFIDF+RandomForest,0.8266,0.8266,0.8264,0.8265,0.9009
Text+Meta LogReg,0.8250,0.8251,0.8251,0.8250,0.9110
TFIDF+LinearSVM,0.8250,0.8251,0.8251,0.8250,0.9093
BoW+LogReg,0.8234,0.8235,0.8235,0.8234,0.9092
TFIDF+NB,0.8141,0.8145,0.8143,0.8141,0.9044
FastText+LogReg,0.8109,0.8115,0.8112,0.8109,0.8990


,TN,FP,FN,TP,FPR,FNR
model,,,,,,
BoW+NB,265,59,64,252,0.182,0.203
BoW+LogReg,265,59,54,262,0.182,0.171
TFIDF+NB,258,66,53,263,0.204,0.168
TFIDF+LogReg,270,54,54,262,0.167,0.171
TFIDF+LinearSVM,264,60,52,264,0.185,0.165
TFIDF+RandomForest,271,53,58,258,0.164,0.184
TFIDF+GradBoost,267,57,53,263,0.176,0.168
Word2Vec+LogReg,255,69,54,262,0.213,0.171
FastText+LogReg,256,68,53,263,0.210,0.168


## 5. Где методы работают хорошо и плохо
Длина текста (test, все модели); размер обучающей выборки и leave-one-domain-out (быстрые модели, список `SLICE`).

In [12]:
bins=pd.qcut(te_df.n_words,4,duplicates='drop'); rows=[]
for k in MODELS:
    for b,g in pd.Series(test_pred[k]==yte).groupby(bins): rows.append((k,str(b),g.mean()))
L=pd.DataFrame(rows,columns=['model','len_bin','acc']).pivot(index='model',columns='len_bin',values='acc').round(4); L.to_csv('output/test_by_length.csv'); display(L)
rng=np.random.default_rng(SEED); lc=[]
for k in SLICE:
    for s in [100,300,1000,len(tr_df)]:
        idx=rng.choice(len(tr_df),s,replace=False); lc.append((k,s,accuracy_score(yte,MODELS[k]().fit(tr_df.iloc[idx],y[idx]).predict(te_df))))
LC=pd.DataFrame(lc,columns=['model','n_train','acc']).pivot(index='model',columns='n_train',values='acc').round(4); LC.to_csv('output/test_learning_curve.csv'); display(LC)
rows=[]
for dm in sorted(df_all.domain.unique()):
    a=df_all[df_all.domain!=dm]; b=df_all[df_all.domain==dm]
    for k in SLICE: rows.append((dm,k,accuracy_score(b.label,MODELS[k]().fit(a,a.label.values).predict(b))))
LODO=pd.DataFrame(rows,columns=['test_domain','model','acc']).pivot(index='model',columns='test_domain',values='acc').round(4); LODO['min']=LODO.min(axis=1); LODO.to_csv('output/leave_one_domain_out.csv'); display(LODO)

len_bin,"(16.999, 36.0]","(36.0, 54.0]","(54.0, 74.0]","(74.0, 108.0]"
model,,,,
BoW+LogReg,0.7396,0.8194,0.8491,0.8917
BoW+NB,0.7219,0.8129,0.8553,0.8471
FastText+LogReg,0.7692,0.7613,0.8616,0.8535
FastText+MLP,0.7160,0.7355,0.7736,0.8408
TFIDF+GradBoost,0.7337,0.7935,0.8868,0.9045
TFIDF+LinearSVM,0.7456,0.8065,0.8742,0.8790
TFIDF+LogReg,0.7515,0.8065,0.8805,0.8917
TFIDF+MLP,0.7633,0.8323,0.8931,0.9363
TFIDF+NB,0.7396,0.8258,0.8302,0.8662


n_train,100,300,1000,2560
model,,,,
BoW+LogReg,0.7547,0.7672,0.8062,0.8234
BoW+NB,0.7422,0.7625,0.8016,0.8078
TFIDF+GradBoost,0.7344,0.7891,0.8141,0.8281
TFIDF+LinearSVM,0.7938,0.7812,0.8312,0.8250
TFIDF+LogReg,0.7719,0.7875,0.8016,0.8312
TFIDF+MLP,0.6234,0.6422,0.7984,0.8500
TFIDF+NB,0.7031,0.7203,0.7891,0.8141
Text+Meta LogReg,0.7734,0.7891,0.8188,0.8250


test_domain,здоровье,политика,спорт,экономика,min
model,,,,,
BoW+LogReg,0.8225,0.8476,0.8356,0.8542,0.8225
BoW+NB,0.8082,0.8728,0.8494,0.8297,0.8082
TFIDF+GradBoost,0.7758,0.8476,0.8243,0.8374,0.7758
TFIDF+LinearSVM,0.8381,0.8728,0.8595,0.8632,0.8381
TFIDF+LogReg,0.8381,0.8728,0.8570,0.8606,0.8381
TFIDF+MLP,0.7890,0.8690,0.8557,0.8503,0.7890
TFIDF+NB,0.7986,0.8640,0.8419,0.8374,0.7986
Text+Meta LogReg,0.8441,0.8451,0.8444,0.8581,0.8441


## 6. Объяснимость (XAI)
SHAP/LIME не установлены, поэтому реализованы методы той же идеи: коэффициенты логистической регрессии, локальная окклюзия слов (аналог LIME), перестановочная важность метаданных.

In [13]:
m=fitted['TFIDF+LogReg']; vec=m.steps[0][1].named_transformers_['t']; clf=m.steps[1][1]; fn=np.array(vec.get_feature_names_out()); w=clf.coef_[0]
xai={'достоверные':list(fn[np.argsort(w)[-15:]][::-1]),'фейк':list(fn[np.argsort(w)[:15]])}; print(xai)
def occlusion(model,s,top=6):
    words=s.split(); base=model.predict_proba(pd.DataFrame({'content':[s]}))[0,1]; d=[]
    for i,wd in enumerate(words): d.append((wd,base-model.predict_proba(pd.DataFrame({'content':[' '.join(words[:i]+words[i+1:])]}))[0,1]))
    return base,sorted(d,key=lambda x:-abs(x[1]))[:top]
i=int(np.where(yte==0)[0][0]); b,d=occlusion(m,te_df.content[i]); print('Пример (истинно фейк):',te_df.content[i][:200]); print('P(достоверная)=%.3f'%b,[(a,round(c,3)) for a,c in d])
mm=fitted['Text+Meta LogReg']; pi=permutation_importance(mm,te_df,yte,n_repeats=20,random_state=SEED,scoring='accuracy')
PI=pd.Series(pi.importances_mean,index=mm.feature_names_in_).round(4); PI.to_csv('output/permutation_importance.csv'); display(PI)

{'достоверные': ['представитель', 'заявил', 'заявил представитель', 'передаёт', 'передаёт агентство', 'агентство', 'согласно', 'согласно статистике', 'статистике', 'пресс', 'сообщила', 'сообщила пресс', 'служба', 'пресс служба', 'данным'], 'фейк': ['сенсация', 'шок', 'разоблачение', 'ужасе', 'врачи ужасе', 'тайный заговор', 'тайный', 'заговор', 'слив', 'информации', 'слив информации', 'изучили пациенты', 'повысил рынок', 'молчат', 'правда']}
Пример (истинно фейк): Вакцина зарегистрировали вирус в этом году . Пациенты выявили больница в этом году. это запретили показывать! Вакцина испытали лекарство в ближайшее время.
P(достоверная)=0.032 [('запретили', np.float64(-0.098)), ('показывать!', np.float64(-0.046)), ('Пациенты', np.float64(-0.032)), ('году', np.float64(-0.029)), ('это', np.float64(-0.021)), ('испытали', np.float64(-0.017))]


title         0.0000
text          0.0000
label         0.0000
domain        0.0000
content       0.2491
n_words       0.0018
n_excl        0.0275
caps_ratio    0.0143
dtype: float64

## 7. Часть B — нейросети и трансформеры (`RUN_DL=True`)
Не исполнялась в среде автора (нет torch). Протокол тот же: train → test; результаты сравнивать с лучшей классической моделью тестом Макнемара.

In [20]:
%pip install transformers


     ---------------------------------------- 0.0/41.6 kB ? eta -:--:--
     ---------------------------------------- 0.0/41.6 kB ? eta -:--:--
     ---------------------------- --------- 30.7/41.6 kB 435.7 kB/s eta 0:00:01
     -------------------------------------- 41.6/41.6 kB 402.4 kB/s eta 0:00:00
     ---------------------------------------- 0.0/57.4 kB ? eta -:--:--
     ---------------------------------------- 0.0/57.4 kB ? eta -:--:--
     ------- -------------------------------- 10.2/57.4 kB ? eta -:--:--
     ------- -------------------------------- 10.2/57.4 kB ? eta -:--:--
     -------------------- ----------------- 30.7/57.4 kB 187.9 kB/s eta 0:00:01
     -------------------------------------- 57.4/57.4 kB 274.0 kB/s eta 0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
if RUN_DL:
    import torch, torch.nn as nn
    from collections import Counter
    from transformers import AutoTokenizer, AutoModel
    dev='cuda' if torch.cuda.is_available() else 'cpu'; torch.manual_seed(SEED)
    cnt=Counter(w for s in tr_df.content for w in tok(s)); vocab={w:i+2 for i,w in enumerate(w for w,c in cnt.items() if c>=2)}; LM=160
    enc=lambda s:(([vocab.get(w,1) for w in tok(s)][:LM])+[0]*LM)[:LM]
    Xtr=torch.tensor([enc(s) for s in tr_df.content]); Xte=torch.tensor([enc(s) for s in te_df.content]); ytr=torch.tensor(y)
    class RNN(nn.Module):
        def __init__(s,cell): super().__init__(); s.e=nn.Embedding(len(vocab)+2,128,padding_idx=0); s.r=cell(128,128,batch_first=True,bidirectional=True); s.o=nn.Linear(256,2)
        def forward(s,x): h,_=s.r(s.e(x)); return s.o(h.max(1).values)
    class TextCNN(nn.Module):
        def __init__(s): super().__init__(); s.e=nn.Embedding(len(vocab)+2,128,padding_idx=0); s.c=nn.ModuleList([nn.Conv1d(128,100,k) for k in (2,3,4)]); s.o=nn.Linear(300,2)
        def forward(s,x): e=s.e(x).transpose(1,2); return s.o(torch.cat([torch.relu(c(e)).max(2).values for c in s.c],1))
    def fit_pred(model,fwd,Xa,ya,Xb,epochs,bs,lr):
        model=model.to(dev); opt=torch.optim.AdamW(model.parameters(),lr=lr)
        for _ in range(epochs):
            model.train(); perm=torch.randperm(len(ya))
            for i in range(0,len(ya),bs):
                idx=perm[i:i+bs]; opt.zero_grad(); nn.functional.cross_entropy(fwd(model,Xa,idx),ya[idx].to(dev)).backward(); opt.step()
        model.eval()
        with torch.no_grad(): lo=torch.cat([fwd(model,Xb,torch.arange(i,min(i+128,len(Xb[0] if isinstance(Xb,tuple) else Xb)))).cpu() for i in range(0,len(Xb[0] if isinstance(Xb,tuple) else Xb),128)])
        return lo.argmax(1).numpy(), torch.softmax(lo,1)[:,1].numpy()
    f_plain=lambda m,X,idx: m(X[idx].to(dev)); dl_rows=[]
    for k,mk in {'LSTM':lambda:RNN(nn.LSTM),'GRU':lambda:RNN(nn.GRU),'CNN':TextCNN}.items():
        p,s=fit_pred(mk(),f_plain,Xtr,ytr,Xte,8,64,1e-3); test_pred[k]=p; dl_rows.append(dict(model=k,**metrics(yte,p,s)))
    BERT='cointegrated/rubert-tiny2'; tk=AutoTokenizer.from_pretrained(BERT)
    E=lambda t: tk(list(t),truncation=True,max_length=256,padding='max_length',return_tensors='pt'); Etr,Ete=E(tr_df.content),E(te_df.content)
    class Head(nn.Module):
        def __init__(s,kind):
            super().__init__(); s.b=AutoModel.from_pretrained(BERT); H=s.b.config.hidden_size; s.kind=kind
            if kind=='cnn': s.c=nn.ModuleList([nn.Conv1d(H,100,k) for k in (2,3,4)]); s.o=nn.Linear(300,2)
            elif kind=='lstm': s.r=nn.LSTM(H,128,batch_first=True,bidirectional=True); s.o=nn.Linear(256,2)
            else: s.o=nn.Linear(H,2)
        def forward(s,ids,att):
            h=s.b(input_ids=ids,attention_mask=att).last_hidden_state
            if s.kind=='cnn': e=h.transpose(1,2); return s.o(torch.cat([torch.relu(c(e)).max(2).values for c in s.c],1))
            if s.kind=='lstm': o,_=s.r(h); return s.o(o.max(1).values)
            return s.o(h[:,0])
    f_bert=lambda m,E_,idx: m(E_['input_ids'][idx].to(dev),E_['attention_mask'][idx].to(dev))
    class W(dict): pass
    for name,kind in {'BERT':'cls','BERT-CNN':'cnn','BERT-LSTM':'lstm'}.items():
        mdl=Head(kind).to(dev); opt=torch.optim.AdamW(mdl.parameters(),lr=3e-5)
        for _ in range(2):
            mdl.train(); perm=torch.randperm(len(y))
            for i in range(0,len(y),16):
                idx=perm[i:i+16]; opt.zero_grad(); nn.functional.cross_entropy(f_bert(mdl,Etr,idx),ytr[idx].to(dev)).backward(); opt.step()
        mdl.eval()
        with torch.no_grad(): lo=torch.cat([f_bert(mdl,Ete,torch.arange(i,min(i+64,len(yte)))).cpu() for i in range(0,len(yte),64)])
        p=lo.argmax(1).numpy(); test_pred[name]=p; dl_rows.append(dict(model=name,**metrics(yte,p,torch.softmax(lo,1)[:,1].numpy())))
    DLT=pd.DataFrame(dl_rows).set_index('model').round(4); DLT.to_csv('output/test_metrics_dl.csv'); display(DLT)
    best=T.index[0]
    for k in DLT.index:
        a=test_pred[k]==yte; b=test_pred[best]==yte; n01=int((a&~b).sum()); n10=int((~a&b).sum()); print(k,'vs',best,'McNemar p =',1.0 if n01+n10==0 else round(stats.binomtest(n01,n01+n10,.5).pvalue,4))
else: print('RUN_DL=False — часть B пропущена')

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 3236.30it/s]
[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 55/55 [00:00<00:00, 2552.78it/s]
[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Stat

KeyboardInterrupt: 

In [27]:
#pd.DataFrame(dl_rows).round(4)
#pd.DataFrame(dl_rows).round(4).to_csv('output/test_metrics_dl.csv')
best=T.index[0]
for k in ['LSTM','GRU','CNN','BERT','BERT-CNN']:
    a=test_pred[k]==yte; b=test_pred[best]==yte; n01=int((a&~b).sum()); n10=int((~a&b).sum())
    print(k,'vs',best,'p =',1.0 if n01+n10==0 else round(stats.binomtest(n01,n01+n10,.5).pvalue,4))

LSTM vs TFIDF+MLP p = 0.0015
GRU vs TFIDF+MLP p = 0.1755
CNN vs TFIDF+MLP p = 0.0649
BERT vs TFIDF+MLP p = 0.4966
BERT-CNN vs TFIDF+MLP p = 0.0789
